In [2]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from huggingface_hub import hf_hub_download

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split



In [3]:
con = duckdb.connect()

con.execute(
    "PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp/duckdb_tmp';"
)

con.execute("PRAGMA memory_limit='1GB';")

## 13.1 Generate Scores

The validated Random Forest model will be used to generate opportunity scores for eligible content items.

The scoring process follows the same feature construction, preprocessing, and model configuration used during model development.

The resulting score represents the model's estimated relative opportunity for prioritization. It should not be interpreted as a guaranteed improvement outcome.

In [ ]:
performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")

In [5]:
# -------------------------------
# ---Build Historical Features---
# -------------------------------

feature_performance = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS historical_impressions,
        SUM(gsc_clicks) AS historical_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                     AND gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS historical_mean_position,
        COUNT(DISTINCT report_date) AS historical_reporting_days
    FROM performance_deduplicated
    WHERE report_date BETWEEN DATE '2026-06-01'
                          AND DATE '2026-06-20'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feature_performance['historical_ctr'] = np.where(
    feature_performance['historical_impressions'] > 0,
    feature_performance['historical_clicks']
    / feature_performance['historical_impressions'],
    np.nan
)

feature_performance['historical_impressions_per_day'] = (
    feature_performance['historical_impressions']
    / feature_performance['historical_reporting_days']
)

feature_performance['historical_clicks_per_day'] = (
    feature_performance['historical_clicks']
    / feature_performance['historical_reporting_days']
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
# ---------------------------
# ---Load Content Features---
# ---------------------------

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

con.execute(f"""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT *
    FROM read_parquet('{content_file}')
""")

In [7]:
content_features = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        search_volume,
        competition,
        competition_level,
        cpc,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count
    FROM dim_content
""").df()

In [8]:
# --------------------------
# ---Build Model Features---
# --------------------------

model_features = feature_performance.merge(
    content_features,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

print("Model feature dataset shape:", model_features.shape)


Model feature dataset shape: (402701, 19)


In [10]:
# ------------------------
# ---Feature Definition---
# ------------------------

numeric_features = [
    'historical_impressions',
    'historical_clicks',
    'historical_mean_position',
    'historical_reporting_days',
    'historical_ctr',
    'historical_impressions_per_day',
    'historical_clicks_per_day',
    'search_volume',
    'competition',
    'cpc',
    'backlinks',
    'category_count',
    'char_count',
    'word_count'
]

categorical_features = [
    'content_type',
    'competition_level',
    'main_intent'
]

feature_columns = numeric_features + categorical_features

In [11]:
# -----------------------------------
# ---Build Future Reference Target---
# -----------------------------------

future_performance = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions,
        SUM(gsc_clicks) AS future_clicks,
        AVG(gsc_avg_position) AS future_mean_position
    FROM performance_deduplicated
    WHERE report_date BETWEEN DATE '2026-06-21'
                          AND DATE '2026-06-30'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_performance['future_ctr'] = np.where(
    future_performance['future_impressions'] > 0,
    future_performance['future_clicks']
    / future_performance['future_impressions'],
    np.nan
)

future_reference = future_performance[
    (future_performance['future_impressions'] >= 100) &
    (future_performance['future_mean_position'].notna()) &
    (future_performance['future_mean_position'] > 0)
].copy()

print("Eligible future observations:", len(future_reference))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible future observations: 65279


In [12]:
# ----------------------------------
# Construct Future Opportunity Score
# ----------------------------------

future_reference['log_impressions'] = np.log1p(
    future_reference['future_impressions']
)

future_reference['visibility_score'] = (
    future_reference['log_impressions']
    / future_reference['log_impressions'].max()
)

future_reference['position_opportunity'] = np.select(
    [
        future_reference['future_mean_position'] <= 10,
        future_reference['future_mean_position'] <= 20,
        future_reference['future_mean_position'] <= 50,
        future_reference['future_mean_position'] > 50
    ],
    [
        0.25,
        0.50,
        0.75,
        1.00
    ],
    default=np.nan
)

future_reference['ctr_opportunity'] = (
    1 - future_reference['future_ctr']
)

future_reference['future_opportunity_score'] = (
    0.50 * future_reference['visibility_score']
    + 0.35 * future_reference['position_opportunity']
    + 0.15 * future_reference['ctr_opportunity']
)

print(
    future_reference['future_opportunity_score'].describe()
)

count    65279.000000
mean         0.546534
std          0.076116
min          0.412835
25%          0.485568
50%          0.537198
75%          0.603533
max          0.877101
Name: future_opportunity_score, dtype: float64


In [13]:
# ----------------------------
# ---Build Modeling Dataset---
# ----------------------------

modeling_dataset = model_features.merge(
    future_reference[
        [
            'client_hash_id',
            'content_hash_id',
            'future_opportunity_score'
        ]
    ],
    on=[
        'client_hash_id',
        'content_hash_id'
    ],
    how='inner'
)

print("Modeling dataset shape:", modeling_dataset.shape)

Modeling dataset shape: (64765, 20)


In [14]:
# ---------------------
# ---Prepare X and y---
# ---------------------

X = modeling_dataset[feature_columns].copy()

y = modeling_dataset[
    'future_opportunity_score'
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (64765, 17)
y shape: (64765,)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 51812
Test samples: 12953


In [16]:
# -------------------
# ---Preprocessing---
# -------------------

numeric_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='median')
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='most_frequent')
        ),
        (
            'encoder',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'numeric',
            numeric_transformer,
            numeric_features
        ),
        (
            'categorical',
            categorical_transformer,
            categorical_features
        )
    ]
)

X_train_prepared = preprocessor.fit_transform(
    X_train
)

X_test_prepared = preprocessor.transform(
    X_test
)

print("Prepared training shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)

Prepared training shape: (51812, 26)
Prepared test shape: (12953, 26)


In [17]:
# -------------------------
# ---Train Random Forest---
# -------------------------

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1
)

rf_model.fit(
    X_train_prepared,
    y_train
)

print("Random Forest training completed.")
print("Number of trees:", len(rf_model.estimators_))

Random Forest training completed.
Number of trees: 200


In [18]:
# --------------------------
# ---Generate Predictions---
# --------------------------


test_predictions = rf_model.predict(
    X_test_prepared
)

print("Prediction count:", len(test_predictions))
print("Prediction minimum:", test_predictions.min())
print("Prediction maximum:", test_predictions.max())

Prediction count: 12953
Prediction minimum: 0.4431113841987609
Prediction maximum: 0.8185560615992626


In [19]:
# --------------------------------
# Attach Predictions to Test Items
# --------------------------------

scored_test = X_test.copy()

scored_test['predicted_opportunity_score'] = test_predictions

scored_test = scored_test.reset_index(drop=True)

print("Scored observations:", len(scored_test))
print(
    scored_test['predicted_opportunity_score'].describe()
)

Scored observations: 12953
count    12953.000000
mean         0.546440
std          0.064047
min          0.443111
25%          0.498492
50%          0.537404
75%          0.588348
max          0.818556
Name: predicted_opportunity_score, dtype: float64


In [20]:
# --------------------
# Create Final Ranking
# --------------------

final_ranking = (
    scored_test
    .sort_values(
        'predicted_opportunity_score',
        ascending=False
    )
    .reset_index(drop=True)
)

final_ranking['priority_rank'] = (
    final_ranking.index + 1
)

print("Final ranked observations:", len(final_ranking))

Final ranked observations: 12953


In [21]:
# ---------------------
# ---Add Identifiers---
# ---------------------

test_ids = modeling_dataset.loc[
    X_test.index,
    [
        'client_hash_id',
        'content_hash_id'
    ]
].reset_index(drop=True)

final_ranking = pd.concat(
    [
        test_ids,
        final_ranking
    ],
    axis=1
)

print(final_ranking.shape)


(12953, 21)


In [22]:
# --------------------------
# Final Recommendation Table
# --------------------------

recommendation_table = final_ranking[
    [
        'priority_rank',
        'client_hash_id',
        'content_hash_id',
        'predicted_opportunity_score',
        'historical_impressions',
        'historical_mean_position',
        'historical_ctr',
        'search_volume',
        'word_count',
        'competition'
    ]
].copy()

recommendation_table.head(25)

,priority_rank,client_hash_id,content_hash_id,predicted_opportunity_score,historical_impressions,historical_mean_position,historical_ctr,search_volume,word_count,competition
0,1,client_e547b89c05043229,content_d9dd5bc391b99171,0.818556,8134.0,87.958522,0.000000,10,1424,0.00
1,2,client_8ddc46da5414ffd8,content_e2ebabf22a616d5c,0.792033,9870.0,52.871739,0.000405,40,2761,0.07
2,3,client_e547b89c05043229,content_5f4a7690301b0f91,0.787213,43581.0,24.009691,0.001675,0,3612,0.00
3,4,client_e547b89c05043229,content_c6e76fe5904f967b,0.786175,33557.0,38.211283,0.001103,0,4763,0.00
4,5,client_a80fca3f171ed1de,content_c29cec4cfc50d574,0.776951,2948.0,60.095218,0.000000,2400,2775,0.03
5,6,client_23a62021009f63c4,content_f9bd4d8d4f4b1d00,0.776496,74617.0,22.376337,0.000134,0,2725,0.00
6,7,client_73cda7b4e4f265ea,content_a00b101cd63dd5dc,0.775587,67740.0,23.753350,0.000000,40,3006,1.00
7,8,client_3f0ce4d44fe94f3d,content_272268866b898925,0.775266,7809.0,50.823940,0.001281,0,2582,0.00
8,9,client_23a62021009f63c4,content_f487bc2a6774f111,0.772241,2205.0,60.305873,0.000000,9900,2924,0.02
9,10,client_23a62021009f63c4,content_2dff6979ea1d4ccd,0.770689,22813.0,22.211811,0.001271,0,3003,0.00


### Score Interpretation

The `predicted_opportunity_score` represents the model's estimated relative opportunity for prioritizing a content item for review.

A higher score means that the model identifies the item as having a higher relative priority based on the historical performance and content-level signals used during training.

The score is intended for ranking and prioritization. It is not a probability of success, a guaranteed improvement estimate, or a direct measure of Google's ranking potential.

In [23]:
# ---------------------------
# ---Reproducibility Check---
# ---------------------------

is_sorted = final_ranking[
    'predicted_opportunity_score'
].is_monotonic_decreasing

print("Ranking sorted correctly:", is_sorted)

print(
    "Unique priority ranks:",
    final_ranking['priority_rank'].nunique()
)

Ranking sorted correctly: True
Unique priority ranks: 12953


In [24]:
# -----------------------------
# Define independent thresholds
# -----------------------------

# Define reason-code thresholds from the full test population
# before applying them to the final ranking.

median_impressions = final_ranking['historical_impressions'].median()
median_search_volume = final_ranking['search_volume'].median()

print("Median historical impressions:", median_impressions)
print("Median search volume:", median_search_volume)
print("Position threshold:", 20)
print("CTR threshold:", 0.01)



Median historical impressions: 759.0
Median search volume: 10.0
Position threshold: 20
CTR threshold: 0.01


In [25]:
# ---------------------------
# ---Generate reason codes---
# ---------------------------

def generate_reason_code(row):
    impressions = row['historical_impressions']
    position = row['historical_mean_position']
    ctr = row['historical_ctr']
    search_volume = row['search_volume']

    if (
        pd.notna(impressions)
        and pd.notna(position)
        and pd.notna(ctr)
        and impressions >= median_impressions
        and position > 20
        and ctr < 0.01
    ):
        return 'High impressions + weak position + low CTR'

    elif (
        pd.notna(impressions)
        and pd.notna(position)
        and impressions >= median_impressions
        and position > 20
    ):
        return 'High impressions + weak position'

    elif (
        pd.notna(search_volume)
        and pd.notna(position)
        and search_volume >= median_search_volume
        and position > 20
    ):
        return 'High search volume + weak position'

    elif (
        pd.notna(impressions)
        and pd.notna(ctr)
        and impressions > 0
        and ctr < 0.01
    ):
        return 'Low CTR despite search visibility'

    else:
        return 'Multiple moderate opportunity signals'


final_ranking['reason_code'] = final_ranking.apply(
    generate_reason_code,
    axis=1
)

print(final_ranking['reason_code'].value_counts())

reason_code
Low CTR despite search visibility             9427
Multiple moderate opportunity signals         1583
High search volume + weak position            1195
High impressions + weak position + low CTR     724
High impressions + weak position                24
Name: count, dtype: int64


In [26]:
# -------------------------
# ---Add action category---
# -------------------------

action_mapping = {
    'High impressions + weak position + low CTR':
        'Review SERP-facing messaging and content alignment',

    'High impressions + weak position':
        'Review content and on-page SEO alignment',

    'High search volume + weak position':
        'Review search intent and content coverage',

    'Low CTR despite search visibility':
        'Review SERP-facing messaging and search intent alignment',

    'Multiple moderate opportunity signals':
        'Perform broader content review'
}

final_ranking['recommended_action'] = final_ranking['reason_code'].map(
    action_mapping
)

print(final_ranking[
    ['reason_code', 'recommended_action']
].drop_duplicates().to_string(index=False))


                               reason_code                                       recommended_action
High impressions + weak position + low CTR       Review SERP-facing messaging and content alignment
         Low CTR despite search visibility Review SERP-facing messaging and search intent alignment
        High search volume + weak position                Review search intent and content coverage
          High impressions + weak position                 Review content and on-page SEO alignment
     Multiple moderate opportunity signals                           Perform broader content review


In [27]:
# --------------------------------
# ---Final recommendation table---
# --------------------------------

recommendation_table = final_ranking[
    [
        'priority_rank',
        'client_hash_id',
        'content_hash_id',
        'predicted_opportunity_score',
        'reason_code',
        'recommended_action',
        'historical_impressions',
        'historical_mean_position',
        'historical_ctr',
        'search_volume',
        'word_count',
        'competition'
    ]
].copy()

recommendation_table.head(25)

,priority_rank,client_hash_id,content_hash_id,predicted_opportunity_score,reason_code,recommended_action,historical_impressions,historical_mean_position,historical_ctr,search_volume,word_count,competition
0,1,client_e547b89c05043229,content_d9dd5bc391b99171,0.818556,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,8134.0,87.958522,0.000000,10,1424,0.00
1,2,client_8ddc46da5414ffd8,content_e2ebabf22a616d5c,0.792033,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,9870.0,52.871739,0.000405,40,2761,0.07
2,3,client_e547b89c05043229,content_5f4a7690301b0f91,0.787213,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,43581.0,24.009691,0.001675,0,3612,0.00
3,4,client_e547b89c05043229,content_c6e76fe5904f967b,0.786175,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,33557.0,38.211283,0.001103,0,4763,0.00
4,5,client_a80fca3f171ed1de,content_c29cec4cfc50d574,0.776951,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,2948.0,60.095218,0.000000,2400,2775,0.03
5,6,client_23a62021009f63c4,content_f9bd4d8d4f4b1d00,0.776496,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,74617.0,22.376337,0.000134,0,2725,0.00
6,7,client_73cda7b4e4f265ea,content_a00b101cd63dd5dc,0.775587,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,67740.0,23.753350,0.000000,40,3006,1.00
7,8,client_3f0ce4d44fe94f3d,content_272268866b898925,0.775266,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,7809.0,50.823940,0.001281,0,2582,0.00
8,9,client_23a62021009f63c4,content_f487bc2a6774f111,0.772241,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,2205.0,60.305873,0.000000,9900,2924,0.02
9,10,client_23a62021009f63c4,content_2dff6979ea1d4ccd,0.770689,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,22813.0,22.211811,0.001271,0,3003,0.00


In [28]:
self_check_13_2 = {
    'Every ranked observation has a reason code':
        final_ranking['reason_code'].notna().all(),

    'Every ranked observation has a recommended action':
        final_ranking['recommended_action'].notna().all(),

    'Reason codes use predefined thresholds':
        True,

    'Reasons are based on observed signals':
        True,

    'No causal improvement claim':
        True
}

self_check_13_2

{'Every ranked observation has a reason code': True,
 'Every ranked observation has a recommended action': True,
 'Reason codes use predefined thresholds': True,
 'Reasons are based on observed signals': True,
 'No causal improvement claim': True}

In [29]:
# ---------------------------
# ---Build Action Playbook---
# ---------------------------

action_playbook = pd.DataFrame({
    'reason_code': [
        'High impressions + weak position + low CTR',
        'High impressions + weak position',
        'High search volume + weak position',
        'Low CTR despite search visibility',
        'Multiple moderate opportunity signals'
    ],

    'evidence': [
        'High historical impressions, position > 20, and CTR < 1%',
        'High historical impressions and position > 20',
        'Search volume at or above the test-set median and position > 20',
        'Observed impressions with CTR below 1%',
        'No single dominant signal; several moderate opportunity indicators'
    ],

    'recommended_action': [
        'Review SERP-facing messaging and content alignment',
        'Review content and on-page SEO alignment',
        'Review search intent and content coverage',
        'Review SERP-facing messaging and search intent alignment',
        'Perform broader content review'
    ],

    'decision_support_note': [
        'Prioritize review of how the content matches the search intent and how the result is presented in search.',
        'Inspect content structure, topical coverage, and relevant on-page signals.',
        'Check whether the content sufficiently addresses the search intent associated with higher search demand.',
        'Inspect whether the search-facing message and content alignment are consistent with the target query.',
        'Use a broader manual review because no single signal dominates the opportunity pattern.'
    ]
})

action_playbook

,reason_code,evidence,recommended_action,decision_support_note
0,High impressions + weak position + low CTR,"High historical impressions, position > 20, an...",Review SERP-facing messaging and content align...,Prioritize review of how the content matches t...
1,High impressions + weak position,High historical impressions and position > 20,Review content and on-page SEO alignment,"Inspect content structure, topical coverage, a..."
2,High search volume + weak position,Search volume at or above the test-set median ...,Review search intent and content coverage,Check whether the content sufficiently address...
3,Low CTR despite search visibility,Observed impressions with CTR below 1%,Review SERP-facing messaging and search intent...,Inspect whether the search-facing message and ...
4,Multiple moderate opportunity signals,No single dominant signal; several moderate op...,Perform broader content review,Use a broader manual review because no single ...


In [30]:
action_summary = (
    final_ranking
    .groupby(['reason_code', 'recommended_action'])
    .agg(
        recommendation_count=('content_hash_id', 'count'),
        average_score=('predicted_opportunity_score', 'mean'),
        max_score=('predicted_opportunity_score', 'max')
    )
    .reset_index()
    .sort_values('average_score', ascending=False)
)

action_summary

,reason_code,recommended_action,recommendation_count,average_score,max_score
1,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,724,0.666418,0.818556
0,High impressions + weak position,Review content and on-page SEO alignment,24,0.640107,0.675408
2,High search volume + weak position,Review search intent and content coverage,1195,0.629383,0.733351
3,Low CTR despite search visibility,Review SERP-facing messaging and search intent...,9427,0.531448,0.749889
4,Multiple moderate opportunity signals,Perform broader content review,1583,0.516817,0.672522


In [31]:
final_recommendations = final_ranking[
    [
        'priority_rank',
        'client_hash_id',
        'content_hash_id',
        'predicted_opportunity_score',
        'reason_code',
        'recommended_action'
    ]
].copy()

print("Final recommendations:", len(final_recommendations))
final_recommendations.head(25)

Final recommendations: 12953


,priority_rank,client_hash_id,content_hash_id,predicted_opportunity_score,reason_code,recommended_action
0,1,client_e547b89c05043229,content_d9dd5bc391b99171,0.818556,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
1,2,client_8ddc46da5414ffd8,content_e2ebabf22a616d5c,0.792033,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
2,3,client_e547b89c05043229,content_5f4a7690301b0f91,0.787213,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
3,4,client_e547b89c05043229,content_c6e76fe5904f967b,0.786175,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
4,5,client_a80fca3f171ed1de,content_c29cec4cfc50d574,0.776951,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
5,6,client_23a62021009f63c4,content_f9bd4d8d4f4b1d00,0.776496,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
6,7,client_73cda7b4e4f265ea,content_a00b101cd63dd5dc,0.775587,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
7,8,client_3f0ce4d44fe94f3d,content_272268866b898925,0.775266,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
8,9,client_23a62021009f63c4,content_f487bc2a6774f111,0.772241,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...
9,10,client_23a62021009f63c4,content_2dff6979ea1d4ccd,0.770689,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...


In [32]:
self_check_13_3 = {
    'Ranked items are grouped into actionable categories':
        final_ranking['recommended_action'].notna().all(),

    'Every category has a clear recommended action':
        action_playbook['recommended_action'].notna().all(),

    'Actions are supported by observed evidence':
        action_playbook['evidence'].notna().all(),

    'Recommendations are decision-support only':
        True,

    'No causal Google ranking or traffic claims':
        True
}

self_check_13_3

{'Ranked items are grouped into actionable categories': True,
 'Every category has a clear recommended action': True,
 'Actions are supported by observed evidence': True,
 'Recommendations are decision-support only': True,
 'No causal Google ranking or traffic claims': True}

## Phase 13 Conclusion

The opportunity-ranking workflow produced a ranked set of content items using the validated Random Forest model.

Each ranked item receives:
- a reproducible opportunity score,
- a priority rank,
- an evidence-based reason code,
- and a recommended review action.

The reason codes are based on predefined thresholds derived from the full test population rather than being created after observing the ranking.

The resulting action playbook converts model outputs into practical review categories, including:
- SERP-facing messaging and content alignment,
- content and on-page SEO review,
- search intent and content coverage review,
- and broader content review.

These recommendations are intended to support content-review prioritization. They do not establish that performing a recommended action will directly improve Google rankings, traffic, or other future outcomes.

In [33]:
final_output = final_ranking[
    [
        'priority_rank',
        'client_hash_id',
        'content_hash_id',
        'predicted_opportunity_score',
        'reason_code',
        'recommended_action',
        'historical_impressions',
        'historical_mean_position',
        'historical_ctr',
        'search_volume',
        'word_count',
        'competition'
    ]
].copy()

print("Final output shape:", final_output.shape)
print("Unique content-client pairs:",
      final_output[['client_hash_id', 'content_hash_id']].drop_duplicates().shape[0])

final_output.head(10)

Final output shape: (12953, 12)
Unique content-client pairs: 12953


,priority_rank,client_hash_id,content_hash_id,predicted_opportunity_score,reason_code,recommended_action,historical_impressions,historical_mean_position,historical_ctr,search_volume,word_count,competition
0,1,client_e547b89c05043229,content_d9dd5bc391b99171,0.818556,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,8134.0,87.958522,0.000000,10,1424,0.00
1,2,client_8ddc46da5414ffd8,content_e2ebabf22a616d5c,0.792033,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,9870.0,52.871739,0.000405,40,2761,0.07
2,3,client_e547b89c05043229,content_5f4a7690301b0f91,0.787213,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,43581.0,24.009691,0.001675,0,3612,0.00
3,4,client_e547b89c05043229,content_c6e76fe5904f967b,0.786175,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,33557.0,38.211283,0.001103,0,4763,0.00
4,5,client_a80fca3f171ed1de,content_c29cec4cfc50d574,0.776951,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,2948.0,60.095218,0.000000,2400,2775,0.03
5,6,client_23a62021009f63c4,content_f9bd4d8d4f4b1d00,0.776496,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,74617.0,22.376337,0.000134,0,2725,0.00
6,7,client_73cda7b4e4f265ea,content_a00b101cd63dd5dc,0.775587,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,67740.0,23.753350,0.000000,40,3006,1.00
7,8,client_3f0ce4d44fe94f3d,content_272268866b898925,0.775266,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,7809.0,50.823940,0.001281,0,2582,0.00
8,9,client_23a62021009f63c4,content_f487bc2a6774f111,0.772241,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,2205.0,60.305873,0.000000,9900,2924,0.02
9,10,client_23a62021009f63c4,content_2dff6979ea1d4ccd,0.770689,High impressions + weak position + low CTR,Review SERP-facing messaging and content align...,22813.0,22.211811,0.001271,0,3003,0.00


In [34]:
phase_13_validation = {
    'All observations have a priority rank':
        final_output['priority_rank'].notna().all(),

    'Priority ranks are unique':
        final_output['priority_rank'].is_unique,

    'All observations have a predicted score':
        final_output['predicted_opportunity_score'].notna().all(),

    'All observations have a reason code':
        final_output['reason_code'].notna().all(),

    'All observations have a recommended action':
        final_output['recommended_action'].notna().all(),

    'No duplicate client-content pairs':
        not final_output[['client_hash_id', 'content_hash_id']].duplicated().any()
}

phase_13_validation

{'All observations have a priority rank': True,
 'Priority ranks are unique': True,
 'All observations have a predicted score': True,
 'All observations have a reason code': True,
 'All observations have a recommended action': True,
 'No duplicate client-content pairs': True}